<a href="https://colab.research.google.com/github/therealsaad/talkora-AI-model-fine-tuning/blob/main/Talkora_Class1_Qwen3_Cloud_QLoRA_Pilot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Talkora — Class 1 adaptive Qwen3: cloud QLoRA pilot

This is a **research-only** fine-tuning run for Qwen/Qwen3-1.7B. It does not alter the live Qwen3-4B, deploy to Talkora, or validate speech.

Before running:

1. Open this notebook in [Google Colab](https://colab.research.google.com/) and select **Runtime → Change runtime type → GPU**. Colab's free GPU access is not guaranteed.
2. On your Windows laptop in the talkora-ai-starter folder, prepare an archive containing *only* the three synthetic dataset files:

       Compress-Archive -Path .\datasets\training\class1-adaptive\train.jsonl,.\datasets\training\class1-adaptive\validation.jsonl,.\datasets\training\class1-adaptive\test.jsonl -DestinationPath .\class1-adaptive-pilot.zip

   If the ZIP exists already, review and rename it before creating another.
3. **Never upload real student speech, personal details, student PINs, .env files, or secrets.**
4. The current examples are synthetic and marked unreviewed. The notebook audits them and requires an explicit pilot-only acknowledgment before training.
5. Download the adapter ZIP when finished. Colab runtimes are temporary.

This notebook uses publicly available model weights and packages. The Colab free tier may offer a GPU subject to changing limits; paid plans require a separate decision.


In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU assigned. Set Runtime > Change runtime type > GPU in Colab."
print("GPU:", torch.cuda.get_device_name(0))
print("Memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
print("PyTorch:", torch.__version__, "CUDA runtime:", torch.version.cuda)


GPU: Tesla T4
Memory (GiB): 14.6
PyTorch: 2.11.0+cu130 CUDA runtime: 13.0


In [ ]:
%pip -q install -U "transformers>=4.51,<5" "peft>=0.15,<1" accelerate bitsandbytes datasets sentencepiece
import transformers, peft, bitsandbytes
print("Transformers:", transformers.__version__, "PEFT:", peft.__version__, "bitsandbytes:", bitsandbytes.__version__)


Transformers: 4.57.6 PEFT: 0.21.2 bitsandbytes: 0.50.2


## Upload and validate the dataset

Upload the ZIP containing train.jsonl, validation.jsonl, and test.jsonl from the local synthetic data generator. The ZIP is inspected in memory; it is not extracted onto the runtime filesystem.


In [ ]:
from google.colab import files
from pathlib import PurePosixPath
import io, zipfile, json, re, random

uploads = files.upload()
assert len(uploads) == 1, "Upload one ZIP file only."
name, content = next(iter(uploads.items()))
assert name.lower().endswith(".zip"), "Expected a ZIP."
part_text = {}
with zipfile.ZipFile(io.BytesIO(content)) as archive:
    for info in archive.infolist():
        if info.is_dir():
            continue
        basename = PurePosixPath(info.filename.replace(chr(92), "/")).name
        assert basename in {"train.jsonl", "validation.jsonl", "test.jsonl"}, "Unexpected ZIP member"
        assert basename not in part_text, "Duplicate ZIP member"
        assert info.file_size < 10_000_000, "Unexpectedly large member"
        part_text[basename] = archive.read(info).decode("utf-8-sig")
assert set(part_text) == {"train.jsonl", "validation.jsonl", "test.jsonl"}

rows = {}
for split in ("train", "validation", "test"):
    split_rows = [json.loads(line) for line in part_text[f"{split}.jsonl"].splitlines() if line.strip()]
    assert split_rows, f"Empty {split} split"
    for row in split_rows:
        msgs = row.get("messages", [])
        assert [m.get("role") for m in msgs] == ["system", "user", "assistant"]
        hint = json.loads(msgs[2]["content"])
        assert set(hint) == {"hint"} and isinstance(hint["hint"], str)
        assert 0 < len(hint["hint"].split()) <= 12
        assert row.get("metadata", {}).get("synthetic") is True, "Non-synthetic examples are prohibited"
        assert row.get("metadata", {}).get("level_id") == "g01-l01"
    rows[split] = split_rows
print("Dataset validated:", {split: len(split_rows) for split, split_rows in rows.items()})
assert sum(len(x) for x in rows.values()) == 240, "Unexpected dataset size; review before continuing"


Saving class1-adaptive-v2-review.zip to class1-adaptive-v2-review (2).zip
Dataset validated: {'train': 192, 'validation': 24, 'test': 24}


In [ ]:
uploads = files.upload()


Saving class1-adaptive-v2-review.zip to class1-adaptive-v2-review (1).zip


In [ ]:
def describe(row):
    user = json.loads(row["messages"][1]["content"])
    hint = json.loads(row["messages"][2]["content"])["hint"]
    return user.get("target"), user.get("transcript"), user.get("scaffold_mode"), user.get("support_language"), hint

model_rows = [r for r in rows["train"] if describe(r)[2] in ("MODEL", "GUIDE")]
generic = [
    describe(r) for r in model_rows
    if describe(r)[0] and not re.search(r"\b" + re.escape(str(describe(r)[0])) + r"\b", describe(r)[4], re.I)
]
print("MODEL/GUIDE examples:", len(model_rows))
print("Examples without the actual target word in the hint:", len(generic))
print("\nExamples needing improvement:")
for item in generic[:12]:
    print(item)
print("\nRandom examples:")
for row in random.Random(2026).sample(rows["train"], min(12, len(rows["train"]))):
    print(describe(row))
print("\nWARNING: Synthetic/unreviewed data, not educationally certified.")


MODEL/GUIDE examples: 96
Examples without the actual target word in the hint: 0

Examples needing improvement:

Random examples:
('an', 'an', 'PRACTISE', 'hi', 'अच्छी कोशिश! अब an से छोटा वाक्य बनाओ।')
('is', 'up', 'MODEL', 'en', 'Our little word is is. Say is slowly.')
('on', "I don't know", 'GUIDE', 'hi', 'मैं बोलती हूँ: on। तुम भी on बोलो।')
('on', 'I can say on.', 'EXTEND', 'hi', 'अच्छी कोशिश! on वाला एक और छोटा वाक्य बोलो।')
('school', "I don't know", 'GUIDE', 'hi', 'मैं बोलती हूँ: school। तुम भी school बोलो।')
('an', 'I can say an.', 'EXTEND', 'en', "Let's use an in another little school sentence.")
('at', 'at', 'PRACTISE', 'mr', 'छान प्रयत्न! आता at वापरून छोटे वाक्य म्हण.')
('up', 'up', 'PRACTISE', 'mr', 'मस्त प्रयत्न! up वापरून वाक्य म्हण.')
('school', 'school', 'PRACTISE', 'en', "Good effort! Let's use school in school talk.")
('up', 'I can say up.', 'EXTEND', 'en', "Nice try using up! Let's make another short sentence.")
('it', "I don't know", 'GUIDE', 'hi', 'कोई बात नहीं। स

## Quality checkpoint

If many MODEL or GUIDE hints do not actually pronounce/model their target word, **fix your local data generator, regenerate the ZIP, and re-upload** before training a serious pilot.

The next cell is an acknowledgment for a **non-production research experiment only**. It does not assert that the dataset has been reviewed or that the model is safe for students.


In [ ]:
approval = input("Type PILOT ONLY to acknowledge the unreviewed synthetic data: ").strip()
assert approval == "PILOT ONLY", "Training cancelled pending your review."
print("Proceeding with an experimental pilot, never a production deployment.")


Type PILOT ONLY to acknowledge the unreviewed synthetic data: PILOT ONLY
Proceeding with an experimental pilot, never a production deployment.


## Train Qwen3-1.7B with 4-bit NF4 + LoRA

The official Hugging Face checkpoint is downloaded the first time (multiple GB). This pilot trains **only the assistant JSON hint**, not the system prompt or student transcript. The 1.7B adapter **cannot be merged into your live 4B model**.


In [ ]:
import os
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

BASE = "Qwen/Qwen3-1.7B"
OUTPUT = "/content/talkora-qwen3-1.7b-class1-v2-ADAPTER-PILOT"
MAX_LENGTH = 768
tokenizer = AutoTokenizer.from_pretrained(BASE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def encode(row):
    messages = row["messages"]
    prefix = tokenizer.apply_chat_template(messages[:2], tokenize=True, add_generation_prompt=True, enable_thinking=False)
    completion = tokenizer.encode(messages[2]["content"] + tokenizer.eos_token, add_special_tokens=False)
    assert len(prefix) + len(completion) <= MAX_LENGTH, "A prompt exceeds MAX_LENGTH. Inspect instead of silently truncating."
    return {
        "input_ids": prefix + completion,
        "attention_mask": [1] * (len(prefix) + len(completion)),
        "labels": [-100] * len(prefix) + completion,
    }

class CoachingDataset(Dataset):
    def __init__(self, examples):
        self.examples = [encode(example) for example in examples]
    def __len__(self):
        return len(self.examples)
    def __getitem__(self, i):
        return self.examples[i]

train_data = CoachingDataset(rows["train"])
validation_data = CoachingDataset(rows["validation"])
print("Max tokenized length:", max(len(x["input_ids"]) for x in train_data.examples))

def pad_batch(batch):
    length = max(len(example["input_ids"]) for example in batch)
    ret = {"input_ids": [], "attention_mask": [], "labels": []}
    for x in batch:
        pad = length - len(x["input_ids"])
        ret["input_ids"].append(x["input_ids"] + [tokenizer.pad_token_id] * pad)
        ret["attention_mask"].append(x["attention_mask"] + [0] * pad)
        ret["labels"].append(x["labels"] + [-100] * pad)
    return {k: torch.tensor(v, dtype=torch.long) for k, v in ret.items()}

quant = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
model = AutoModelForCausalLM.from_pretrained(
    BASE, quantization_config=quant, torch_dtype=torch.float16, device_map={"": 0}
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
model = get_peft_model(
    model,
    LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
               lora_dropout=0.05, bias="none", task_type="CAUSAL_LM")
)
model.print_trainable_parameters()
args = TrainingArguments(
    output_dir=OUTPUT, num_train_epochs=1,
    per_device_train_batch_size=1, per_device_eval_batch_size=1,
    gradient_accumulation_steps=4, learning_rate=1e-4,
    fp16=True, gradient_checkpointing=True, optim="paged_adamw_8bit",
    eval_strategy="epoch", save_strategy="no", logging_steps=10,
    report_to="none", remove_unused_columns=False,
)
trainer = Trainer(model=model, args=args, train_dataset=train_data,
                  eval_dataset=validation_data, data_collator=pad_batch)
trainer.train()
print("Validation:", trainer.evaluate())
os.makedirs(OUTPUT, exist_ok=True)
model.save_pretrained(OUTPUT)
tokenizer.save_pretrained(OUTPUT)
print("Saved research adapter:", OUTPUT)


Max tokenized length: 382


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 3,211,264 || all params: 1,723,786,240 || trainable%: 0.1863


Epoch,Training Loss,Validation Loss
1,1.293400,1.183995


Validation: {'eval_loss': 1.1839953660964966, 'eval_runtime': 4.0124, 'eval_samples_per_second': 5.982, 'eval_steps_per_second': 5.982, 'epoch': 1.0}
Saved research adapter: /content/talkora-qwen3-1.7b-class1-v2-ADAPTER-PILOT


In [ ]:
import shutil
from google.colab import files

path = shutil.make_archive(
    "/content/Talkora_Class1_Qwen3_1.7B_ADAPTER_V2_PILOT",
    "zip",
    OUTPUT
)

files.download(path)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import json
from collections import Counter

print("TEST WORDS:", Counter(
    json.loads(r["messages"][1]["content"])["target"]
    for r in rows["test"]
))

print("FIRST TEST:", json.loads(
    rows["test"][0]["messages"][1]["content"]
)["target"])

print("ADAPTER:", OUTPUT)

TEST WORDS: Counter({'on': 3, 'up': 3, 'school': 3, 'walk': 3, 'am': 2, 'an': 2, 'at': 2, 'in': 2, 'is': 2, 'it': 2})
FIRST TEST: am
ADAPTER: /content/talkora-qwen3-1.7b-class1-v2-ADAPTER-PILOT


In [ ]:
%run -i /content/Talkora_Class1_24Case_Colab_Evaluation.py

Completed 4/24 held-out comparisons
Completed 8/24 held-out comparisons
Completed 12/24 held-out comparisons
Completed 16/24 held-out comparisons
Completed 20/24 held-out comparisons
Completed 24/24 held-out comparisons
{
  "base_model": "Qwen/Qwen3-1.7B",
  "test_cases": 24,
  "baseline": {
    "missing_target_in_model_or_guide": 2,
    "passed_automated_checks": 21,
    "hint_length": 1
  },
  "adapter": {
    "invalid_json_contract": 9,
    "passed_automated_checks": 13,
    "hint_length": 1,
    "missing_target_in_model_or_guide": 1
  },
  "limitations": [
    "Synthetic and not human-reviewed; automated checks do not establish teaching quality.",
    "Transcript visibility is not an acoustic pronunciation or intelligence score.",
    "Human educators must review all rows, including correct JSON responses."
  ]
}

Sample comparisons (original Qwen -> trained adapter):
#1 am / MODEL: {"hint":"Say it with a gentle tone."} -> {"hint": "am. am. am. am. am. am. am. am. am. am. am. am. a

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Examine held-out responses

Inspect whether the model produces valid JSON, the right word, and safe hints. **Five examples cannot validate teaching quality.** Keep test cases separate from training and conduct a teacher-reviewed evaluation before deployment.


In [ ]:
model.eval()
for item in rows["test"][:5]:
    prompt = tokenizer.apply_chat_template(item["messages"][:2], tokenize=True,
                                            add_generation_prompt=True, enable_thinking=False)
    inputs = torch.tensor([prompt], dtype=torch.long, device="cuda")
    with torch.no_grad():
        generated = model.generate(input_ids=inputs, attention_mask=torch.ones_like(inputs),
                                   max_new_tokens=60, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    result = tokenizer.decode(generated[0, len(prompt):], skip_special_tokens=True).strip()
    print("Target / student transcript / mode / language:", describe(item)[:4])
    print("Generated:", result)
    try:
        value = json.loads(result)
        print("Valid JSON:", set(value) == {"hint"} and isinstance(value["hint"], str) and len(value["hint"].split()) <= 12)
    except (ValueError, TypeError):
        print("Valid JSON: False")
    print("---")


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Target / student transcript / mode / language: ('up', 'at', 'MODEL', 'en')
Generated: {"hint": "Let's say it together. Up, up, up."}
Valid JSON: True
---
Target / student transcript / mode / language: ('up', 'at', 'MODEL', 'en')
Generated: {"hint": "Let's try again."}
Valid JSON: True
---
Target / student transcript / mode / language: ('up', "I don't know", 'GUIDE', 'en')
Generated: {"hint": "Let's try again. I'll say the word up."}
Valid JSON: True
---
Target / student transcript / mode / language: ('up', "I don't know", 'GUIDE', 'en')
Generated: {"hint": "Let's try again."}
Valid JSON: True
---
Target / student transcript / mode / language: ('up', 'up', 'PRACTISE', 'en')
Generated: {"hint": "Let's try again. I'm here to help you say the word up with confidence."}
Valid JSON: False
---


In [ ]:
from google.colab import files
files.upload()


Saving Talkora_Class1_24Case_Colab_Evaluation.py to Talkora_Class1_24Case_Colab_Evaluation (1).py


{'Talkora_Class1_24Case_Colab_Evaluation (1).py': b'"""Paste into a NEW CODE CELL after the pilot notebook\'s training/evaluation cells.\n\nRequires the existing Colab variables: model, tokenizer, rows, torch.\nDoes not train, modify, save, or deploy the adapter.\n"""\n\nimport csv\nimport io\nimport json\nimport re\nimport zipfile\nfrom collections import Counter\n\nfrom google.colab import files\n\nassert len(rows["test"]) == 24, "Expected 24 held-out examples. Check dataset."\nassert hasattr(model, "disable_adapter"), "Expected loaded PEFT model from the pilot."\nmodel.eval()\n\n\ndef generate_coach_hint(case, use_adapter):\n    prompt_tokens = tokenizer.apply_chat_template(\n        case["messages"][:2],\n        tokenize=True,\n        add_generation_prompt=True,\n        enable_thinking=False,\n    )\n    input_ids = torch.tensor([prompt_tokens], dtype=torch.long, device="cuda")\n\n    def _generate():\n        with torch.inference_mode():\n            output = model.generate(\n 

In [ ]:
print("Dataset:", "rows" in globals())
print("Model:", "model" in globals())
print("Tokenizer:", "tokenizer" in globals())

Dataset: True
Model: True
Tokenizer: True


In [ ]:
%run -i /content/Talkora_Class1_24Case_Colab_Evaluation.py


Completed 4/24 held-out comparisons
Completed 8/24 held-out comparisons
Completed 12/24 held-out comparisons
Completed 16/24 held-out comparisons
Completed 20/24 held-out comparisons
Completed 24/24 held-out comparisons
{
  "base_model": "Qwen/Qwen3-1.7B",
  "test_cases": 24,
  "baseline": {
    "passed_automated_checks": 24
  },
  "adapter": {
    "passed_automated_checks": 8,
    "missing_target_in_model_or_guide": 7,
    "hint_length": 1,
    "unhelpful_retry_after_visible_target": 2,
    "invalid_json_contract": 7
  },
  "limitations": [
    "Synthetic and not human-reviewed; automated checks do not establish teaching quality.",
    "Transcript visibility is not an acoustic pronunciation or intelligence score.",
    "Human educators must review all rows, including correct JSON responses."
  ]
}

Sample comparisons (original Qwen -> trained adapter):
#1 up / MODEL: {"hint":"Say the word 'up' with a gentle tone."} -> {"hint": "Let's say it together. Up, up, up."}
#2 up / MODEL: {"hin

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import json
from collections import Counter

print("TEST WORDS:", Counter(
    json.loads(r["messages"][1]["content"])["target"]
    for r in rows["test"]
))

print("FIRST TEST:", json.loads(
    rows["test"][0]["messages"][1]["content"]
)["target"])

print("ADAPTER:", OUTPUT)

In [ ]:
%run Talkora_Class1_24Case_Colab_Evaluation.py


NameError: name 'rows' is not defined

In [ ]:
import shutil
archive_path = shutil.make_archive("/content/Talkora_Class1_Qwen3_1.7B_ADAPTER_PILOT", "zip", OUTPUT)
print("Keep this research-only adapter private and label its base model as:", BASE)
files.download(archive_path)


Keep this research-only adapter private and label its base model as: Qwen/Qwen3-1.7B


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Handoff checklist

- The adapter is for Qwen3-1.7B only; do not attach it to the existing Qwen3-4B Ollama model.
- Do not publish or deploy directly from Colab.
- Conduct a held-out evaluation against untrained Qwen3-1.7B, across successful, struggling, Hindi, Marathi, and adversarial student-speech cases.
- Confirm short spoken-friendly hints, lesson-boundary compliance, and no unsupported pronunciation or IQ claims.
- Keep student data private and secure; use synthetic examples until your school and guardians' privacy requirements are handled.
